In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor


from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import clone

import optuna

import os
import pickle
import warnings

# Suppress warnings to keep the output clean during training
warnings.filterwarnings('ignore')


/home/malloy/Desktop/workspace/Data-Science/Zindi-Competitions/amini soil prediction challenge/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_advanced_features(df):
    """
    Create additional, more advanced features that might help the model performance.

    Args:
        df (pd.DataFrame): The input DataFrame with base features.

    Returns:
        pd.DataFrame: The DataFrame with newly engineered features.
    """
    print("\n--- Creating Advanced Features ---")
    df_enhanced = df.copy()

    # --- 2. Interaction Terms for Key Soil Properties and Climate Variables ---
    # Define sets of features for interaction
    key_soil_features = ['pH', 'soc20', 'cec20', 'BulkDensity', 'ph20']
    key_climate_features = ['bio1', 'bio12', 'lstd', 'lstn'] # Mean Annual Temp, Annual Precip, Day/Night LST
    key_spatial_features = ['lat', 'lon'] # For interaction with geographic coordinates

    # Generate interactions between selected soil features
    # Iterate through unique pairs to avoid duplicates (e.g., A_B vs B_A) and self-interactions
    for i in range(len(key_soil_features)):
        for j in range(i + 1, len(key_soil_features)):
            f1 = key_soil_features[i]
            f2 = key_soil_features[j]
            if f1 in df_enhanced.columns and f2 in df_enhanced.columns:
                df_enhanced[f'{f1}_{f2}_interaction'] = df_enhanced[f1] * df_enhanced[f2]
    print("  Soil feature interactions created.")

    # Generate interactions between soil and climate features
    for soil_f in key_soil_features:
        for climate_f in key_climate_features:
            if soil_f in df_enhanced.columns and climate_f in df_enhanced.columns:
                df_enhanced[f'{soil_f}_{climate_f}_interaction'] = df_enhanced[soil_f] * df_enhanced[climate_f]
    print("  Soil-climate feature interactions created.")

    # Generate interactions between spatial features and other key features
    # This requires 'lat' and 'lon' to be present in the DataFrame.
    # Note: If 'lat' and 'lon' are dropped during climate zone normalization, these features won't be created.
    for geo_f in key_spatial_features:
        if geo_f in df_enhanced.columns:
            for feature in ['pH', 'soc20', 'bio1', 'BulkDensity']: # Example features for geo interactions
                if feature in df_enhanced.columns:
                    df_enhanced[f'{geo_f}_{feature}_interaction'] = df_enhanced[geo_f] * df_enhanced[feature]
    print("  Geographical interaction features created.")


    # --- 3. Polynomial Features for Important Continuous Variables ---
    # Apply polynomial transformation (e.g., squaring) to capture non-linear relationships.
    # Be mindful of adding too many, which can lead to multicollinearity and overfitting.
    polynomial_features_list = [
        'pH', 'soc20', 'cec20', 'BulkDensity', 'bio1', 'bio12', 'lstd', 'lstn',
        'mdem', 'slope', 'tim', # Topographical features
        'dows', 'ls',           # Water features
        'alb', 'wp'             # Land cover features
    ]
    # Filter to only include columns that actually exist in the DataFrame
    polynomial_features_list = [f for f in polynomial_features_list if f in df_enhanced.columns]

    degree = 2 # Creating quadratic features

    for feature in polynomial_features_list:
        df_enhanced[f'{feature}_sq'] = df_enhanced[feature] ** degree
        # You could add more degrees if empirical analysis suggests a higher-order relationship
        # df_enhanced[f'{feature}_cub'] = df_enhanced[feature] ** 3
    print(f"  Polynomial features (degree {degree}) created.")


    # --- 4. Vegetation Index Combinations/Differences (Advanced) ---
    # Assuming MODIS/Landsat/Sentinel mean features are available.
    # Create differences or further ratios between related vegetation indices if they exist and make sense.
    # Example: Difference between two types of NDVI or EVI from different sensors.
    vi_pairs_for_diff = [
        ('MCD43A4_NDVI_mcd43a4_mean', 'MOD09GA_NDVI_mod09ga_mean'),
        ('MCD43A4_EVI_mcd43a4_mean', 'MOD09GA_EVI_mod09ga_mean'),
        ('MOD13Q1_NDVI_scaled_mean', 'S2_NDVI_sent2_mean')
    ]
    for vi1, vi2 in vi_pairs_for_diff:
        if vi1 in df_enhanced.columns and vi2 in df_enhanced.columns:
            df_enhanced[f'{vi1.split("_mean")[0]}_minus_{vi2.split("_mean")[0]}'] = df_enhanced[vi1] - df_enhanced[vi2]
    print("  Vegetation Index differences created.")


    # --- Final NaN Handling for Newly Created Features ---
    # After creating new features, some NaNs might appear (e.g., if original features had NaNs that weren't caught, or edge cases).
    # This step ensures all new features are imputed before returning the DataFrame.
    print("  Performing final NaN imputation for newly created features...")
    new_features_cols = [col for col in df_enhanced.columns if col not in df.columns]
    for col in new_features_cols:
        if df_enhanced[col].isnull().any():
            if df_enhanced[col].dtype in ['float64', 'int64']:
                mean_val = df_enhanced[col].mean()
                if np.isnan(mean_val): # If the entire new column became NaN (e.g., all inputs were NaN)
                    mean_val = 0 # Fallback to 0 or another sensible default
                    print(f"    Warning: Newly created feature '{col}' is entirely NaN. Filling with 0.")
                df_enhanced[col].fillna(mean_val, inplace=True)
            else:
                # For non-numeric new columns (unlikely for these features),
                # consider df_enhanced[col].mode()[0] for categorical or a specific placeholder.
                pass # For this context, we mostly expect numeric features.
    print("  Final NaN imputation for new features complete.")
    print("--- Advanced Features Creation Complete ---")
    return df_enhanced


In [3]:
def load_and_preprocess_data(path='../dataset/', target_variables=None, modis_data_dir=None, apply_scaling=True, apply_climate_normalization=True, n_climate_zones=5):
    """
    Loads the datasets and performs comprehensive preprocessing steps,
    including merging, handling missing values (NaN and Inf), climate zone normalization,
    advanced feature engineering, and general scaling.

    Args:
        path (str): The base directory where the dataset files are located.
        target_variables (list): List of target variable column names to exclude from preprocessing (e.g., from scaling).
        modis_data_dir (str, optional): Path to the directory containing processed MODIS/Landsat/Sentinel parquet files.
                                        If None, these additional datasets are not loaded.
        apply_scaling (bool): Whether to apply StandardScaler to numerical features after climate normalization.
        apply_climate_normalization (bool): Whether to apply climate zone-specific normalization.
        n_climate_zones (int): Number of clusters for KMeans for climate zone identification.

    Returns:
        tuple: A tuple containing preprocessed train_df, test_df, train_gap_df, and test_gap_df.
    """
    print("--- Loading and Preprocessing Data ---")

    # Initialize stage_transformers to store preprocessing information
    stage_transformers = {
        'imputation_means': {}, # Stores means from training data for consistent imputation
        'scaler': None, # For general scaling
        'scaled_columns': [],
        'climate_kmeans_model': None, # Stores KMeans model for climate zones
        'climate_zone_scalers': {}, # Stores StandardScaler per climate zone
    }

    # Load main datasets
    train_path = os.path.join(path, 'Train.csv')
    test_path = os.path.join(path, 'Test.csv')
    gap_train_path = os.path.join(path, 'Gap_Train.csv')
    gap_test_path = os.path.join(path, 'Gap_Test.csv')

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    train_gap_df = pd.read_csv(gap_train_path)
    test_gap_df = pd.read_csv(gap_test_path)

    # Merge BulkDensity into test_gap_df, crucial for later calculations
    test_gap_df = pd.merge(test_gap_df, test_df[['PID', 'BulkDensity']], on='PID', how='left')

    # Handle missing values in initial train/test CSVs with mean imputation
    print("Handling initial missing values (mean imputation)...")
    for column in train_df.columns:
        if train_df[column].isnull().any():
            mean_value = train_df[column].mean()
            train_df[column].fillna(mean_value, inplace=True)
            stage_transformers['imputation_means'][column] = mean_value

    for column in test_df.columns:
        if test_df[column].isnull().any():
            fill_value = stage_transformers['imputation_means'].get(column, test_df[column].mean()) # Use train mean or test mean
            test_df[column].fillna(fill_value, inplace=True)
    print("Initial missing values handled.")

    # --- Load and process multiple auxiliary datasets from parquet files ---
    if modis_data_dir:
        print(f"Loading additional data from {modis_data_dir}...")
        auxiliary_products_info = {
            'MCD43A4': {
                'prefix': 'modis',
                'cols': ['NDVI_mcd43a4', 'EVI_mcd43a4', 'SAVI_mcd43a4', 'GNDVI_mcd43a4',
                         'SR_mcd43a4', 'NDBR_mcd43a4', 'GRVI_mcd43a4', 'Brightness_Index_mcd43a4',
                         'Red_Green_Ratio_mcd43a4', 'Chlorophyll_Index_mcd43a4', 'Blue_NIR_Ratio_mcd43a4']
            },
            'MOD09GA': {
                'prefix': 'modis',
                'cols': ['sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03', 'sur_refl_b04', 'sur_refl_b05',
                         'sur_refl_b06', 'sur_refl_b07', 'NDVI_mod09ga', 'EVI_mod09ga', 'SAVI_mod09ga',
                         'NDWI_mod09ga', 'BSI_mod09ga', 'GEMI_mod09ga', 'ARVI_mod09ga', 'SIPI_mod09ga']
            },
            'MOD11A1': {
                'prefix': 'modis',
                'cols': ['LST_Day_1km', 'LST_Night_1km']
            },
            'MOD13Q1': {
                'prefix': 'modis',
                'cols': ['season_sin', 'season_cos', 'EVI_scaled', 'NDVI_scaled', 'SAVI_m13q1',
                         'MSAVI_m13q1', 'SR_m13q1', 'NDBR_m13q1', 'NDSWIR_m13q1', 'NDSWIR_NIR_m13q1',
                         'Brightness_Index_m13q1', 'Red_Blue_Ratio_m13q1', 'SWIR_Blue_Ratio_m13q1',
                         'Chlorophyll_Red_Edge_m13q1', 'NRI_approx_m13q1', 'PSRI_m13q1', 'SIPI_m13q1', 'MSI_m13q1']
            },
            'MOD16A2': {
                'prefix': 'modis',
                'cols': ['ET', 'PET', 'ESI_mod16a2', 'ETD_mod16a2']
            },
            'L8': {
                'prefix': 'landsat',
                'cols': ['LST_Celsius', 'NDVI_ls8', 'EVI_ls8', 'SAVI_ls8', 'NDWI_ls8', 'BSI_ls']
            },
            'S1': {
                'prefix': 'sentinel',
                'cols': ['VH_to_VV_Ratio_dB', 'VV_minus_VH_dB', 'Span_dB']
            },
            'S2': {
                'prefix': 'sentinel',
                'cols': ['NDVI_sent2', 'EVI_sent2', 'SAVI_sent2', 'NDRE1_sent2', 'CIre_sent2',
                         'NDWI_sent2', 'LSWI_sent2', 'BSI_sent2', 'NDRE2_sent2',
                         'CLOUDY_PIXEL_PERCENTAGE', 'NODATA_PIXEL_PERCENTAGE']
            }
        }

        for product_name, info in auxiliary_products_info.items():
            file_prefix = info['prefix']
            cols_to_aggregate = info['cols']

            file_path = os.path.join(modis_data_dir, f'processed_{file_prefix}_{product_name.lower()}.parquet')

            try:
                product_df = pd.read_parquet(file_path)

                if 'PID' not in product_df.columns:
                    print(f"Warning: 'PID' column not found in {product_name} data ({file_path}). Skipping merge for this product.")
                    continue

                valid_cols = [col for col in cols_to_aggregate if col in product_df.columns]
                if not valid_cols:
                    print(f"Warning: No valid columns to aggregate found in {product_name} ({file_path}). Skipping merge for this product.")
                    continue

                product_aggregated = product_df.groupby('PID')[valid_cols].mean().reset_index()
                rename_dict = {col: f'{product_name}_{col}_mean' for col in valid_cols}
                product_aggregated.rename(columns=rename_dict, inplace=True)

                train_df = pd.merge(train_df, product_aggregated, on='PID', how='left')
                test_df = pd.merge(test_df, product_aggregated, on='PID', how='left')

                for col_orig, col_renamed in rename_dict.items():
                    if col_renamed in train_df.columns:
                        if train_df[col_renamed].isnull().any():
                            mean_val_merged = train_df[col_renamed].mean()
                            train_df[col_renamed].fillna(mean_val_merged, inplace=True)
                            stage_transformers['imputation_means'][col_renamed] = mean_val_merged
                        if test_df[col_renamed].isnull().any():
                            fill_val_merged = stage_transformers['imputation_means'].get(col_renamed, test_df[col_renamed].mean())
                            test_df[col_renamed].fillna(fill_val_merged, inplace=True)
                print(f"{product_name} data loaded and merged successfully from {file_path}.")

            except FileNotFoundError:
                print(f"Error: {product_name} file not found at {file_path}. Skipping this product.")
            except Exception as e:
                print(f"An error occurred while processing {product_name} data from {file_path}: {e}. Skipping this product.")
    else:
        print("Auxiliary data directory not provided. Skipping loading of these datasets.")

    # Set default target variables if not provided
    if target_variables is None:
        target_variables = []

    # Identify numerical columns for initial NaN/Inf handling
    numerical_cols_pre_fe = [col for col in train_df.columns
                             if train_df[col].dtype in ['int64', 'float64']
                             and col != 'PID' and col != 'site'
                             and col not in target_variables]

    print("\nPerforming initial NaN/Inf check and robust imputation...")
    for col in numerical_cols_pre_fe:
        # Convert any +/- inf to NaN
        if (np.isinf(train_df[col]).any()):
            train_df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
        if (np.isinf(test_df[col]).any()):
            test_df[col].replace([np.inf, -np.inf], np.nan, inplace=True)
        
        # Fill remaining NaNs using the mean from the training set (or 0 if all NaN)
        if train_df[col].isnull().any():
            mean_val = train_df[col].mean()
            if np.isnan(mean_val):
                mean_val = 0
                print(f"  Warning: Column '{col}' in train_df is entirely NaN after inf-to-nan. Filling with 0.")
            train_df[col].fillna(mean_val, inplace=True)
            stage_transformers['imputation_means'][col] = mean_val # Store for test_df
        
        if test_df[col].isnull().any():
            fill_val = stage_transformers['imputation_means'].get(col, 0)
            test_df[col].fillna(fill_val, inplace=True)
    print("Initial NaN/Inf check and imputation complete.")

    # --- Apply Advanced Feature Engineering ---
    print("\nApplying advanced feature engineering to train and test data...")
    train_df = create_advanced_features(train_df)
    test_df = create_advanced_features(test_df)
    print("Advanced feature engineering complete.")


    # --- Climate Zone Normalization (if enabled) ---
    if apply_climate_normalization and 'lat' in train_df.columns and 'lon' in train_df.columns:
        print(f"\nApplying climate zone normalization with {n_climate_zones} zones using 'lat' and 'lon'...")
        coords_train = train_df[['lat', 'lon']].values
        coords_test = test_df[['lat', 'lon']].values

        kmeans = KMeans(n_clusters=n_climate_zones, random_state=42, n_init=10)
        train_df['climate_zone'] = kmeans.fit_predict(coords_train)
        test_df['climate_zone'] = kmeans.predict(coords_test)
        stage_transformers['climate_kmeans_model'] = kmeans

        # Define features to apply climate-zone specific scaling
        # This list should include features that are sensitive to climate and were NOT
        # newly created features that might already be scaled in some way.
        # Ensure 'lat' and 'lon' are NOT in this list if they are used as features directly,
        # as normalizing them by zone might remove their raw positional information.
        climate_sensitive_features = [
            'bio1', 'bio12', 'bio15', 'bio7', 'lstd', 'lstn',
            'MOD11A1_LST_Day_1km_mean', 'MOD11A1_LST_Night_1km_mean',
            'L8_LST_Celsius_mean',
            'MCD43A4_NDVI_mcd43a4_mean', 'MCD43A4_EVI_mcd43a4_mean',
            'MOD09GA_NDVI_mod09ga_mean', 'MOD09GA_EVI_mod09ga_mean',
            'MOD13Q1_EVI_scaled_mean', 'MOD13Q1_NDVI_scaled_mean',
            'S2_NDVI_sent2_mean', 'S2_EVI_sent2_mean'
        ]
        # Filter to only include those that actually exist in the dataframe and are numerical
        numerical_cols_after_fe = [col for col in train_df.columns
                                   if train_df[col].dtype in ['int64', 'float64']
                                   and col != 'PID' and col != 'site'
                                   and col not in target_variables] # Exclude target vars

        climate_sensitive_features = [col for col in climate_sensitive_features if col in numerical_cols_after_fe]


        if climate_sensitive_features:
            for zone in sorted(train_df['climate_zone'].unique()):
                zone_mask_train = train_df['climate_zone'] == zone
                zone_mask_test = test_df['climate_zone'] == zone

                if not train_df.loc[zone_mask_train, climate_sensitive_features].empty:
                    scaler = StandardScaler()
                    train_df.loc[zone_mask_train, climate_sensitive_features] = \
                        scaler.fit_transform(train_df.loc[zone_mask_train, climate_sensitive_features])

                    if not test_df.loc[zone_mask_test, climate_sensitive_features].empty:
                        test_df.loc[zone_mask_test, climate_sensitive_features] = \
                            scaler.transform(test_df.loc[zone_mask_test, climate_sensitive_features])
                    stage_transformers['climate_zone_scalers'][zone] = scaler
                    print(f"  Normalized features in climate zone {zone}.")
                else:
                    print(f"  No data for climate zone {zone} in training data for climate-sensitive features. Skipping normalization for this zone.")
            print("Climate zone normalization complete.")
        else:
            print("No climate-sensitive features identified or present for climate zone normalization.")
        train_df.drop(columns=['climate_zone'], errors='ignore', inplace=True)
        test_df.drop(columns=['climate_zone'], errors='ignore', inplace=True)
    elif apply_climate_normalization:
        print("Skipping climate zone normalization: 'lat' or 'lon' not found in data.")


    # Identify numerical columns for final general scaling (after climate normalization if applied)
    # This list will include all original numerical features PLUS the newly created ones.
    numerical_cols_for_general_scaling = [col for col in train_df.columns
                                          if train_df[col].dtype in ['int64', 'float64']
                                          and col != 'PID' and col != 'site'
                                          and col not in target_variables]


    # Feature Scaling (General StandardScaler)
    if apply_scaling and numerical_cols_for_general_scaling:
        print("Applying general StandardScaler to all numerical features (after all FE and climate normalization)...")

        cols_to_scale = numerical_cols_for_general_scaling

        if cols_to_scale:
            scaler = StandardScaler()

            train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])
            test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])

            stage_transformers['scaler'] = scaler
            stage_transformers['scaled_columns'] = cols_to_scale
            print(f"Scaled {len(cols_to_scale)} numerical features.")
        else:
            print("No numerical features available for general scaling.")
            stage_transformers['scaler'] = None
            stage_transformers['scaled_columns'] = []
    elif apply_scaling and not numerical_cols_for_general_scaling:
        print("No numerical features found for general scaling.")
        stage_transformers['scaler'] = None
        stage_transformers['scaled_columns'] = []
    else:
        print("General scaling not applied as per 'apply_scaling=False'.")


    print("Data loading and comprehensive preprocessing complete.")
    print(f"Final train_df shape: {train_df.shape}")
    print(f"Final test_df shape: {test_df.shape}")

    return train_df, test_df, train_gap_df, test_gap_df


In [4]:
def train_and_save_models(X_train, y_train, X_val, y_val, X_test_final, target_columns, models_dir='trained_models'):
    """
    Trains multiple regression models for multi-output prediction using cross-validation,
    saves them, and uses a VotingRegressor to make final predictions on the test set.

    Args:
        X_train (pd.DataFrame): Training features.
        y_train (pd.DataFrame): Training targets.
        X_val (pd.DataFrame): Validation features.
        y_val (pd.DataFrame): Validation targets.
        X_test_final (pd.DataFrame): Test features for making final predictions.
        target_columns (list): A list of target column names (e.g., ['N', 'P', ...]).
        models_dir (str): The directory where trained models will be saved.

    Returns:
        tuple: A tuple containing:
            - trained_models (dict): Dictionary of all trained MultiOutputRegressor models.
            - model_rmse_scores (dict): Dictionary of RMSE scores for each model on the validation set.
            - final_predictions_voting (np.ndarray): Final predictions from the VotingRegressor on X_test_final.
            - voting_regressor (MultiOutputRegressor): The trained VotingRegressor model.
    """
    # Create the directory to save models if it doesn't exist
    if not os.path.exists(models_dir):
        os.makedirs(models_dir)

    # Combine training and validation data for cross-validation
    X_combined = pd.concat([X_train, X_val], axis=0, ignore_index=True)
    y_combined = pd.concat([y_train, y_val], axis=0, ignore_index=True)

    # Set up cross-validation strategy
    cv_folds = KFold(n_splits=5, shuffle=True, random_state=17)

    # Define the base regression models.
    # n_estimators is set to 500 for ensemble models.
    # verbose is set to 0 or -1 for CatBoost and LightGBM to suppress verbose training output.
    # n_jobs=-1 utilizes all available CPU cores for parallel processing where supported.
    base_models = {
        'XGBoost': XGBRegressor(random_state=17, n_estimators=400, learning_rate=0.01, verbosity=0, n_jobs=-1),
        'LightGBM': lgb.LGBMRegressor(random_state=17, n_estimators=450, learning_rate=0.03, verbose=-1, n_jobs=-1),
        # 'CatBoost': CatBoostRegressor(learning_rate=0.01, depth=10, random_state=17, verbose=0),
        'DecisionTree': DecisionTreeRegressor(random_state=17),
        'RandomForest': RandomForestRegressor(random_state=17, n_jobs=-1)
    }

    trained_models = {}        # Stores the trained MultiOutputRegressor instances
    model_rmse_scores = {}     # Stores RMSE scores for each model on the validation set
    voting_estimators = []     # List of (name, estimator) tuples for the VotingRegressor

    print("\n--- Starting Individual Model Training with Cross-Validation ---")
    for name, model in base_models.items():
        print(f"Training {name}...") # Changed to training for brevity

        # Wrap each base model with MultiOutputRegressor to handle multiple target variables
        multi_output_model = MultiOutputRegressor(model)

        # Perform cross-validation evaluation
        cv_scores = cross_val_score(multi_output_model, X_combined, y_combined,
                                   cv=cv_folds, scoring='neg_mean_squared_error', n_jobs=-1)
        cv_rmse = np.sqrt(-cv_scores.mean())
        cv_rmse_std = np.sqrt(-cv_scores).std()

        # Train the model on the combined data for better performance
        multi_output_model.fit(X_combined, y_combined)
        trained_models[name] = multi_output_model

        # Save the trained model to a pickle file for later use
        model_filename = os.path.join(models_dir, f'{name}_model.pkl')
        with open(model_filename, 'wb') as file:
            pickle.dump(multi_output_model, file)
        print(f"Saved {name} model to {model_filename}")

        # Evaluate the model on the validation set and calculate RMSE
        y_val_pred = multi_output_model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
        model_rmse_scores[name] = rmse

        print(f"{name} Cross-Validation RMSE: {cv_rmse:.4f} (+/- {cv_rmse_std * 2:.4f})")
        print(f"{name} Validation Set RMSE: {rmse:.4f}\n")

        # Append the BASE model to voting_estimators.
        # The VotingRegressor expects the raw estimators, not the MultiOutputRegressor wrapped ones.
        voting_estimators.append((name, model))

    # Identify and print the best performing individual model based on RMSE
    best_model_name = min(model_rmse_scores, key=model_rmse_scores.get)
    best_rmse = model_rmse_scores[best_model_name]
    print(f"--- Best individual model: {best_model_name} with RMSE: {best_rmse:.4f} ---")

    # Train the VotingRegressor on all the individual models.
    # It aggregates the predictions of the individual estimators.
    # Wrapped again in MultiOutputRegressor for multi-target prediction.
    print("\n--- Training VotingRegressor (Ensemble Model) with Cross-Validation ---")
    # Here, VotingRegressor receives the base estimators, and then MultiOutputRegressor
    # ensures it handles the multi-target output.
    voting_regressor = MultiOutputRegressor(VotingRegressor(estimators=voting_estimators, n_jobs=-1))

    # Evaluate VotingRegressor with cross-validation
    cv_scores = cross_val_score(voting_regressor, X_combined, y_combined,
                               cv=cv_folds, scoring='neg_mean_squared_error', n_jobs=-1)
    voting_cv_rmse = np.sqrt(-cv_scores.mean())
    voting_cv_rmse_std = np.sqrt(-cv_scores).std()
    print(f"VotingRegressor Cross-Validation RMSE: {voting_cv_rmse:.4f} (+/- {voting_cv_rmse_std * 2:.4f})")

    # Train the voting regressor on the combined data
    voting_regressor.fit(X_combined, y_combined)
    print("VotingRegressor trained successfully.")

    # Make final predictions on the provided test features using the trained VotingRegressor
    final_predictions_voting = voting_regressor.predict(X_test_final)
    print("--- Model Training and Prediction Phase Complete ---")

    return trained_models, model_rmse_scores, final_predictions_voting, voting_regressor


In [5]:
# def tune_and_train_models(X_train, y_train, X_val, y_val, X_test_final, target_columns, 
#                          models_dir='trained_models', use_pretuned_params=False, pretuned_params=None):
#     """
#     Performs hyperparameter tuning for multiple regression models using cross-validation,
#     saves the best models, and creates a VotingRegressor ensemble for final predictions.
    
#     Can either run Optuna optimization or use pre-tuned parameters.

#     Args:
#         X_train (pd.DataFrame): Training features.
#         y_train (pd.DataFrame): Training targets.
#         X_val (pd.DataFrame): Validation features.
#         y_val (pd.DataFrame): Validation targets.
#         X_test_final (pd.DataFrame): Test features for making final predictions.
#         target_columns (list): A list of target column names (e.g., ['N', 'P', 'K', ...]).
#         models_dir (str): The directory where trained models will be saved.
#         use_pretuned_params (bool): If True, uses pretuned_params instead of running Optuna.
#         pretuned_params (dict): Dictionary of pre-tuned parameters for each model.
#                                Format: {'XGBoost': {'n_estimators': 400}, 'LightGBM': {...}, ...}

#     Returns:
#         tuple: A tuple containing:
#             - trained_models (dict): Dictionary of all trained MultiOutputRegressor models.
#             - model_rmse_scores (dict): Dictionary of RMSE scores for each model on the validation set.
#             - final_predictions_voting (np.ndarray): Final predictions from the VotingRegressor on X_test_final.
#             - voting_regressor (MultiOutputRegressor): The trained VotingRegressor model.
#     """
#     # Create the directory to save models if it doesn't exist
#     if not os.path.exists(models_dir):
#         os.makedirs(models_dir)

#     # Define Optuna objective functions for each model (only used if use_pretuned_params=False)
#     def create_objective_function(model_name, base_model, X_data, y_data, cv_folds):
#         def objective(trial):
#             # Define hyperparameter search spaces for each model
#             if model_name == 'XGBoost':
#                 params = {
#                     'n_estimators': trial.suggest_int('n_estimators', 250, 600, step=50)
#                 }
#             elif model_name == 'LightGBM':
#                 params = {
#                     'n_estimators': trial.suggest_categorical('n_estimators', [250, 300, 400, 450, 500, 550])
#                 }
#             elif model_name == 'CatBoost':
#                 params = {
#                     'depth': trial.suggest_categorical('depth', [8, 10])
#                 }
#             elif model_name == 'AdaBoost':
#                 params = {
#                     'n_estimators': trial.suggest_categorical('n_estimators', [50, 100]),
#                     'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.1, 0.5, 1.0])
#                 }
#             elif model_name == 'DecisionTree':
#                 max_depth_choices = [3, 5, 7, 8, 10, 12, None]
#                 max_depth = trial.suggest_categorical('max_depth', max_depth_choices)
#                 params = {'max_depth': max_depth}
#             elif model_name == 'RandomForest':
#                 params = {
#                     'n_estimators': trial.suggest_int('n_estimators', 100, 700, step=100)
#                 }
            
#             # Update model parameters
#             model_copy = clone(base_model)
#             model_copy.set_params(**params)
            
#             # Wrap with MultiOutputRegressor
#             multi_output_model = MultiOutputRegressor(model_copy)
            
#             # Perform cross-validation
#             cv_scores = cross_val_score(multi_output_model, X_data, y_data, 
#                                       cv=cv_folds, scoring='neg_mean_squared_error', n_jobs=-1)
            
#             # Return mean CV score (Optuna minimizes, so we return the negative MSE)
#             return cv_scores.mean()
        
#         return objective

#     # Define the base regression models with default hyperparameters
#     base_models = {
#         'XGBoost': XGBRegressor(random_state=17, learning_rate=0.01, verbosity=0, n_jobs=-1),
#         'LightGBM': lgb.LGBMRegressor(random_state=17, learning_rate=0.03, verbose=-1, n_jobs=-1),
#         'CatBoost': CatBoostRegressor(learning_rate=0.01, random_state=17, depth=10, verbose=0),
#         'DecisionTree': DecisionTreeRegressor(random_state=17),
#         'RandomForest': RandomForestRegressor(random_state=17, n_jobs=-1),
#     }

#     trained_models = {}        # Stores the trained MultiOutputRegressor instances
#     model_rmse_scores = {}     # Stores RMSE scores for each model on the validation set
#     voting_estimators = []     # List of (name, estimator) tuples for the VotingRegressor

#     # Combine training and validation data for cross-validation
#     X_combined = pd.concat([X_train, X_val], axis=0, ignore_index=True)
#     y_combined = pd.concat([y_train, y_val], axis=0, ignore_index=True)

#     # Set up cross-validation strategy
#     cv_folds = KFold(n_splits=5, shuffle=True, random_state=17)

#     if use_pretuned_params and pretuned_params:
#         print("\n--- Training Models with Pre-tuned Parameters and Optuna for Missing Models ---")
        
#         # First, train models with pre-tuned parameters
#         for name, model in base_models.items():
#             if name in pretuned_params:
#                 print(f"Training {name} with pre-tuned parameters: {pretuned_params[name]}")
                
#                 # Create model with pre-tuned parameters
#                 best_model_instance = clone(model)
#                 best_model_instance.set_params(**pretuned_params[name])
                
#                 # Wrap with MultiOutputRegressor and train on full combined data
#                 best_model = MultiOutputRegressor(best_model_instance)
#                 best_model.fit(X_combined, y_combined)
                
#                 trained_models[name] = best_model

#                 # Save the trained model to a pickle file for later use
#                 model_filename = os.path.join(models_dir, f'{name}_tuned_model.pkl')
#                 with open(model_filename, 'wb') as file:
#                     pickle.dump(best_model, file)
#                 print(f"Saved {name} tuned model to {model_filename}")

#                 # Evaluate the model on the validation set and calculate RMSE
#                 y_val_pred = best_model.predict(X_val)
#                 rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
#                 model_rmse_scores[name] = rmse
#                 print(f"{name} Validation Set RMSE: {rmse:.4f}\n")

#                 # Extract the base estimator for voting
#                 if hasattr(best_model, 'estimator'):
#                     base_estimator = best_model.estimator
#                 else:
#                     base_estimator = best_model.estimators_[0]
                
#                 voting_estimators.append((name, base_estimator))
        
#         # Now train models without pre-tuned parameters using Optuna
#         models_to_tune = [name for name in base_models.keys() if name not in pretuned_params]
#         if models_to_tune:
#             print(f"\n--- Running Optuna tuning for models without pre-tuned parameters: {models_to_tune} ---")
#             for name in models_to_tune:
#                 model = base_models[name]
#                 print(f"Training and tuning {name} with Optuna and cross-validation...")
                
#                 # Create Optuna study for this model
#                 study = optuna.create_study(
#                     direction='maximize',  # Maximize negative MSE (minimize MSE)
#                     sampler=optuna.samplers.TPESampler(seed=17),
#                     pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
#                 )
                
#                 # Create objective function for this model
#                 objective_func = create_objective_function(name, model, X_combined, y_combined, cv_folds)
                
#                 # Optimize hyperparameters
#                 study.optimize(objective_func, n_trials=20, show_progress_bar=False)
                
#                 # Get best parameters and create best model
#                 best_params = study.best_params
#                 best_model_instance = clone(model)
#                 best_model_instance.set_params(**best_params)
                
#                 # Wrap with MultiOutputRegressor and train on full combined data
#                 best_model = MultiOutputRegressor(best_model_instance)
#                 best_model.fit(X_combined, y_combined)
                
#                 trained_models[name] = best_model

#                 # Save the trained model to a pickle file for later use
#                 model_filename = os.path.join(models_dir, f'{name}_tuned_model.pkl')
#                 with open(model_filename, 'wb') as file:
#                     pickle.dump(best_model, file)
#                 print(f"Saved {name} tuned model to {model_filename}")

#                 # Evaluate the model on the validation set and calculate RMSE
#                 y_val_pred = best_model.predict(X_val)
#                 rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
#                 model_rmse_scores[name] = rmse
                
#                 # Also report cross-validation score from Optuna
#                 cv_rmse = np.sqrt(-study.best_value)  # Convert back from negative MSE to RMSE
#                 print(f"{name} Best Cross-Validation RMSE: {cv_rmse:.4f}")
#                 print(f"{name} Validation Set RMSE: {rmse:.4f}")
#                 print(f"Best parameters: {best_params}")
#                 print(f"Number of trials completed: {len(study.trials)}\n")

#                 # Extract the base estimator for voting (the actual model, not MultiOutputRegressor)
#                 if hasattr(best_model, 'estimator'):
#                     base_estimator = best_model.estimator
#                 else:
#                     base_estimator = best_model.estimators_[0]
                
#                 voting_estimators.append((name, base_estimator))
    
#     else:
#         print("\n--- Starting Individual Model Training with Optuna Hyperparameter Tuning and Cross-Validation ---")
#         for name, model in base_models.items():
#             print(f"Training and tuning {name} with Optuna and cross-validation...")
            
#             # Create Optuna study for this model
#             study = optuna.create_study(
#                 direction='maximize',  # Maximize negative MSE (minimize MSE)
#                 sampler=optuna.samplers.TPESampler(seed=17),
#                 pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
#             )
            
#             # Create objective function for this model
#             objective_func = create_objective_function(name, model, X_combined, y_combined, cv_folds)
            
#             # Optimize hyperparameters
#             study.optimize(objective_func, n_trials=20, show_progress_bar=False)
            
#             # Get best parameters and create best model
#             best_params = study.best_params
#             best_model_instance = clone(model)
#             best_model_instance.set_params(**best_params)
            
#             # Wrap with MultiOutputRegressor and train on full combined data
#             best_model = MultiOutputRegressor(best_model_instance)
#             best_model.fit(X_combined, y_combined)
            
#             trained_models[name] = best_model

#             # Save the trained model to a pickle file for later use
#             model_filename = os.path.join(models_dir, f'{name}_tuned_model.pkl')
#             with open(model_filename, 'wb') as file:
#                 pickle.dump(best_model, file)
#             print(f"Saved {name + 'tuned'} model to {model_filename}")

#             # Evaluate the model on the validation set and calculate RMSE
#             y_val_pred = best_model.predict(X_val)
#             rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
#             model_rmse_scores[name] = rmse
            
#             # Also report cross-validation score from Optuna
#             cv_rmse = np.sqrt(-study.best_value)  # Convert back from negative MSE to RMSE
#             print(f"{name} Best Cross-Validation RMSE: {cv_rmse:.4f}")
#             print(f"{name} Validation Set RMSE: {rmse:.4f}")
#             print(f"Best parameters: {best_params}")
#             print(f"Number of trials completed: {len(study.trials)}\n")

#             # Extract the base estimator for voting (the actual model, not MultiOutputRegressor)
#             if hasattr(best_model, 'estimator'):
#                 base_estimator = best_model.estimator
#             else:
#                 base_estimator = best_model.estimators_[0]
            
#             voting_estimators.append((name, base_estimator))

#     # Identify and print the best performing individual model based on RMSE
#     if model_rmse_scores:  # Only if we have models trained
#         best_model_name = min(model_rmse_scores, key=model_rmse_scores.get)
#         best_rmse = model_rmse_scores[best_model_name]
#         print(f"--- Best individual model: {best_model_name} with RMSE: {best_rmse:.4f} ---")

#         # Train the VotingRegressor on the combined data for better ensemble performance
#         print("\n--- Training VotingRegressor (Ensemble Model) with Cross-Validation ---")
#         voting_regressor = MultiOutputRegressor(VotingRegressor(estimators=voting_estimators, n_jobs=-1))
        
#         # Evaluate VotingRegressor with cross-validation
#         cv_scores = cross_val_score(voting_regressor, X_combined, y_combined, 
#                                    cv=cv_folds, scoring='neg_mean_squared_error', n_jobs=-1)
#         voting_cv_rmse = np.sqrt(-cv_scores.mean())
#         print(f"VotingRegressor Cross-Validation RMSE: {voting_cv_rmse:.4f} (+/- {np.sqrt(-cv_scores).std() * 2:.4f})")
        
#         # Train the voting regressor on the combined data
#         voting_regressor.fit(X_combined, y_combined)
#         print("VotingRegressor trained successfully.")

#         # Save the voting regressor
#         voting_filename = os.path.join(models_dir, 'voting_regressor.pkl')
#         with open(voting_filename, 'wb') as file:
#             pickle.dump(voting_regressor, file)
#         print(f"Saved VotingRegressor to {voting_filename}")

#         # Make final predictions on the provided test features using the trained VotingRegressor
#         final_predictions_voting = voting_regressor.predict(X_test_final)
#         print("--- Model Training and Prediction Phase Complete ---")

#         return trained_models, model_rmse_scores, final_predictions_voting, voting_regressor
#     else:
#         print("No models were trained successfully.")
#         return {}, {}, None, None


# # Example usage with pre-tuned parameters:
# """
# # Define your pre-tuned parameters
# pretuned_params = {
#     'XGBoost': {'n_estimators': 400},
#     'LightGBM': {'n_estimators': 450},
#     'CatBoost': {'depth': 10},
#     'AdaBoost': {'n_estimators': 100, 'learning_rate': 0.1},
#     'DecisionTree': {'max_depth': 8},
#     'RandomForest': {'n_estimators': 500}
# }

# # Call the function with pre-tuned parameters
# trained_models, model_rmse_scores, final_predictions_voting, voting_regressor = tune_and_train_models(
#     X_train, y_train, X_val, y_val, X_test_final, target_columns,
#     models_dir='trained_models',
#     use_pretuned_params=True,
#     pretuned_params=pretuned_params
# )
# """

In [6]:
def create_submission_file(test_df, predictions, test_gap_df, target_columns, output_filename='submission.csv'):
    """
    Generates the final submission CSV file in the required format.
    It takes the raw model predictions and transforms them into 'ID' and 'Gap' columns.

    Args:
        test_df (pd.DataFrame): The original test DataFrame (used primarily for 'PID').
        predictions (np.ndarray): The raw predictions from the model (e.g., from the VotingRegressor).
        test_gap_df (pd.DataFrame): The preprocessed test_gap_df, which should contain
                                     'Required' and 'BulkDensity' columns.
        target_columns (list): A list of target column names (e.g., ['N', 'P', 'K', ...]).
        output_filename (str): The desired name for the output CSV submission file.
    """
    print(f"\n--- Creating Submission File: {output_filename} ---")

    # Basic check to ensure predictions align with expected structure
    if predictions.shape[1] != len(target_columns):
        print(f"Warning: Prediction array columns ({predictions.shape[1]}) do not match target column count ({len(target_columns)}).")
    if predictions.shape[0] != test_df.shape[0]:
        print(f"Warning: Prediction array rows ({predictions.shape[0]}) do not match test_df rows ({test_df.shape[0]}).")

    # Convert raw predictions into a pandas DataFrame, associating them with PIDs
    predictions_df = pd.DataFrame(predictions, columns=target_columns)
    predictions_df['PID'] = test_df['PID']

    # Melt the predictions_df from wide format to long format.
    # This creates columns for 'PID', 'Nutrient' (e.g., 'N', 'P'), and 'Available_Nutrients_in_ppm'.
    submission_melted = predictions_df.melt(
        id_vars=['PID'],
        var_name='Nutrient',
        value_name='Available_Nutrients_in_ppm'
    )
    # Sort by PID for consistency and reset index
    submission_melted = submission_melted.sort_values('PID').reset_index(drop=True)

    # Merge the melted predictions with the test_gap_df.
    # This brings in the 'Required' nutrient levels and 'BulkDensity' needed for calculations.
    nutrient_df = pd.merge(test_gap_df, submission_melted, on=['PID', 'Nutrient'], how='left')

    # Calculate 'Available_Nutrients_in_kg_ha' based on the provided formula.
    # soil_depth is constant at 20 cm as per the notebook.
    soil_depth = 20  # cm
    nutrient_df['Available_Nutrients_in_kg_ha'] = (
        nutrient_df['Available_Nutrients_in_ppm'] * soil_depth * nutrient_df['BulkDensity'] * 0.1
    )

    # Calculate the 'Gap' which is the difference between 'Required' and 'Available'.
    # A positive gap means the nutrient needs to be added, negative means there's an excess.
    nutrient_df['Gap'] = nutrient_df['Required'] - nutrient_df['Available_Nutrients_in_kg_ha']

    # Create the unique 'ID' column by concatenating 'PID' and 'Nutrient'.
    nutrient_df['ID'] = nutrient_df['PID'].astype(str) + "_" + nutrient_df['Nutrient']

    # Select only the 'ID' and 'Gap' columns for the final submission file.
    final_submission_df = nutrient_df[['ID', 'Gap']]

    # Save the final DataFrame to a CSV file without the DataFrame index.
    final_submission_df.to_csv(output_filename, index=False)
    print(f"Submission file saved successfully as {output_filename}")
    print(f"First 5 rows of the generated submission file:\n", final_submission_df.head())
    print("--- Submission File Creation Complete ---")


In [7]:
# --- Main execution block ---
if __name__ == "__main__":
    # Define the path to your dataset
    DATASET_PATH = '../dataset/'
    # Path to the directory containing processed auxiliary data files
    PROCESSED_DATA_DIR = '../processed_data/'

    TARGET_COLUMNS = ['N', 'P', 'K', 'Ca', 'Mg', 'S', 'Fe', 'Mn', 'Zn', 'Cu', 'B']

    # Step 1: Load and preprocess data using the enhanced function
    train_df_full, test_df_full, train_gap_df, test_gap_df = load_and_preprocess_data(
        path=DATASET_PATH,
        target_variables=TARGET_COLUMNS,
        modis_data_dir=PROCESSED_DATA_DIR,
        apply_scaling=True, # Set to False if you don't want scaling
    )

    # Define base feature sets
    core_features = [
        'pH', 'ph20', 'BulkDensity', 'cec20', 'ecec20', 'hp20',
        'snd20', 'soc20', 'xhp20'
    ]
    climate_features = [
        'bio1', 'bio12', 'bio15', 'bio7', 'lstd', 'lstn'
    ]
    topographical_features = [
        'mdem', 'slope', 'tim'
    ]
    water_features = [
        'dows', 'ls'
    ]
    land_cover_features = [
        'alb', 'wp'
    ]
    modis16a2_features = [
        'MOD16A2_ET_mean', 'MOD16A2_PET_mean', 'MOD16A2_ESI_mod16a2_mean', 'MOD16A2_ETD_mod16a2_mean'
    ]
    mcd43a4_features = [
        'MCD43A4_NDVI_mcd43a4_mean', 'MCD43A4_EVI_mcd43a4_mean', 'MCD43A4_SAVI_mcd43a4_mean',
        'MCD43A4_GNDVI_mcd43a4_mean', 'MCD43A4_SR_mcd43a4', 'MCD43A4_NDBR_mcd43a4_mean',
        'MCD43A4_GRVI_mcd43a4_mean', 'MCD43A4_BrightnessIndex_mcd43a4_mean',
        'MCD43A4_Red_Green_Ratio_mcd43a4_mean', 'MCD43A4_Chlorophyll_Index_mcd43a4_mean',
        'MCD43A4_Blue_NIR_Ratio_mcd43a4_mean'
    ]
    mod09ga_features = [
        'MOD09GA_sur_refl_b01_mean', 'MOD09GA_sur_refl_b02_mean', 'MOD09GA_sur_refl_b03_mean',
        'MOD09GA_sur_refl_b04_mean', 'MOD09GA_sur_refl_b05_mean', 'MOD09GA_sur_refl_b06_mean',
        'MOD09GA_sur_refl_b07_mean', 'MOD09GA_NDVI_mod09ga_mean', 'MOD09GA_EVI_mod09ga_mean',
        'MOD09GA_SAVI_mod09ga_mean', 'MOD09GA_NDWI_mod09ga_mean', 'MOD09GA_BSI_mod09ga_mean',
        'MOD09GA_GEMI_mod09ga_mean', 'MOD09GA_ARVI_mod09ga_mean', 'MOD09GA_SIPI_mod09ga_mean'
    ]
    mod13q1_features = [
        'MOD13Q1_season_sin_mean', 'MOD13Q1_season_cos_mean', 'MOD13Q1_EVI_scaled_mean',
        'MOD13Q1_NDVI_scaled_mean', 'MOD13Q1_SAVI_m13q1_mean', 'MOD13Q1_MSAVI_m13q1_mean',
        'MOD13Q1_SR_m13q1_mean', '_meanMOD13Q1NDBR_m13q1_mean', 'MOD13Q1_NDSWIR_m13q1_mean',
        'MOD13Q1_NDSWIR_NIR_m13q1_mean', 'MOD13Q1_Brightness_Index_m13q1_mean',
        'MOD13Q1_Red_Blue_Ratio_m13q1_mean', 'MOD13Q1_SWIR_Blue_Ratio_m13q1_mean',
        'MOD13Q1_Chlorophyll_Red_Edge_m13q1_mean', 'MOD13Q1_NRI_approx_m13q1_mean',
        'MOD13Q1_PSRI_m13q1_mean', 'MOD13Q1_SIPI_m13q1_mean', 'MOD13Q1_MSI_m13q1_mean'
    ]
    landsat8_features = [
        'L8_LST_Celsius_mean', 'L8_NDVI_ls8_mean', 'L8_EVI_ls8_mean',
        'L8_SAVI_ls8_mean', 'L8_NDWI_ls8_mean', 'L8_BSI_ls8_mean'
    ]
    sentinel1_features = [
        'S1_VH_to_VV_Ratio_dB_mean', 'S1_VV_minus_VH_dB_mean', 'S1_Span_dB_mean'
    ]
    sentinel2_features = [
        'S2_NDVI_sent2_mean', 'S2_EVI_sent2_mean', 'S2_SAVI_sent2_mean',
        'S2_NDRE1_sent2_mean', 'S2_CIre_sent2_mean', 'S2_NDWI_sent2_mean',
        'S2_LSWI_sent2_mean', 'S2_BSI_sent2_mean', 'S2_NDRE2_sent2_mean',
        'S2_CLOUDY_PIXEL_PERCENTAGE_mean', 'S2_NODATA_PIXEL_PERCENTAGE_mean'
    ]

    advanced_features_generated = [
        'pH_soc20_interaction', 'pH_cec20_interaction', 'pH_BulkDensity_interaction', 'pH_ph20_interaction',
        'soc20_cec20_interaction', 'soc20_BulkDensity_interaction', 'soc20_ph20_interaction',
        'cec20_BulkDensity_interaction', 'cec20_ph20_interaction', 'BulkDensity_ph20_interaction',
        'pH_bio1_interaction', 'pH_bio12_interaction', 'pH_lstd_interaction', 'pH_lstn_interaction',
        'soc20_bio1_interaction', 'soc20_bio12_interaction', 'soc20_lstd_interaction', 'soc20_lstn_interaction',
        'cec20_bio1_interaction', 'cec20_bio12_interaction', 'cec20_lstd_interaction', 'cec20_lstn_interaction',
        'BulkDensity_bio1_interaction', 'BulkDensity_bio12_interaction', 'BulkDensity_lstd_interaction', 'BulkDensity_lstn_interaction',
        'ph20_bio1_interaction', 'ph20_bio12_interaction', 'ph20_lstd_interaction', 'ph20_lstn_interaction',
        'lat_pH_interaction', 'lat_soc20_interaction', 'lat_bio1_interaction', 'lat_BulkDensity_interaction',
        'lon_pH_interaction', 'lon_soc20_interaction', 'lon_bio1_interaction', 'lon_BulkDensity_interaction',
        'pH_sq', 'soc20_sq', 'cec20_sq', 'BulkDensity_sq', 'bio1_sq', 'bio12_sq', 'lstd_sq', 'lstn_sq',
        'mdem_sq', 'slope_sq', 'tim_sq', 'dows_sq', 'ls_sq', 'alb_sq', 'wp_sq',
        'MCD43A4_NDVI_mcd43a4_minus_MOD09GA_NDVI_mod09ga',
        'MCD43A4_EVI_mcd43a4_minus_MOD09GA_EVI_mod09ga',
        'MOD13Q1_NDVI_scaled_minus_S2_NDVI_sent2'
    ]

    all_available_features_master_list = [] # This list will accumulate features for each scenario

    # --- Scenario 1: Core Features ---
    # print("\n--- Running Scenario 1: Core Features ---")
    all_available_features_master_list.extend(core_features)
    
    # selected_features_s1 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s1 = train_df_full[selected_features_s1]
    # y_s1 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s1 = test_df_full[selected_features_s1]
    
    # X_train_s1, X_val_s1, y_train_s1, y_val_s1 = train_test_split(X_s1, y_s1, test_size=0.2, random_state=42)
    
    # _, _, final_predictions_s1, _ = train_and_save_models(
    #     X_train_s1, y_train_s1, X_val_s1, y_val_s1, X_test_final_s1,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario1_core'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s1, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario1_core.csv'
    # )
    # print("Scenario 1 complete.")


    # --- Scenario 2: Core + Climate Features ---
    # print("\n--- Running Scenario 2: Core + Climate Features ---")
    all_available_features_master_list.extend(climate_features)

    # selected_features_s2 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s2 = train_df_full[selected_features_s2]
    # y_s2 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s2 = test_df_full[selected_features_s2]

    # X_train_s2, X_val_s2, y_train_s2, y_val_s2 = train_test_split(X_s2, y_s2, test_size=0.2, random_state=42)

    # _, _, final_predictions_s2, _ = train_and_save_models(
    #     X_train_s2, y_train_s2, X_val_s2, y_val_s2, X_test_final_s2,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario2_core_climate'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s2, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario2_core_climate.csv'
    # )
    # print("Scenario 2 complete.")


    # --- Scenario 3: Core + Climate + Topographical Features ---
    # print("\n--- Running Scenario 3: Core + Climate + Topographical Features ---")
    all_available_features_master_list.extend(topographical_features)

    # selected_features_s3 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s3 = train_df_full[selected_features_s3]
    # y_s3 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s3 = test_df_full[selected_features_s3]

    # X_train_s3, X_val_s3, y_train_s3, y_val_s3 = train_test_split(X_s3, y_s3, test_size=0.2, random_state=42)

    # _, _, final_predictions_s3, _ = train_and_save_models(
    #     X_train_s3, y_train_s3, X_val_s3, y_val_s3, X_test_final_s3,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario3_core_climate_topo'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s3, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario3_core_climate_topo.csv'
    # )
    # print("Scenario 3 complete.")


    # --- Scenario 4: Core + Climate + Topographical + Water Features ---
    # print("\n--- Running Scenario 4: Core + Climate + Topographical + Water Features ---")
    all_available_features_master_list.extend(water_features)

    # selected_features_s4 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s4 = train_df_full[selected_features_s4]
    # y_s4 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s4 = test_df_full[selected_features_s4]

    # X_train_s4, X_val_s4, y_train_s4, y_val_s4 = train_test_split(X_s4, y_s4, test_size=0.2, random_state=42)

    # _, _, final_predictions_s4, _ = train_and_save_models(
    #     X_train_s4, y_train_s4, X_val_s4, y_val_s4, X_test_final_s4,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario4_core_climate_topo_water'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s4, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario4_core_climate_topo_water.csv'
    # )
    # print("Scenario 4 complete.")


    # --- Scenario 5: Core + Climate + Topographical + Water + Land Cover Features ---
    # print("\n--- Running Scenario 5: Core + Climate + Topographical + Water + Land Cover Features ---")
    all_available_features_master_list.extend(land_cover_features)

    # selected_features_s5 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s5 = train_df_full[selected_features_s5]
    # y_s5 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s5 = test_df_full[selected_features_s5]

    # X_train_s5, X_val_s5, y_train_s5, y_val_s5 = train_test_split(X_s5, y_s5, test_size=0.2, random_state=42)

    # _, _, final_predictions_s5, _ = train_and_save_models(
    #     X_train_s5, y_train_s5, X_val_s5, y_val_s5, X_test_final_s5,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario5_core_climate_topo_water_landcover'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s5, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario5_core_climate_topo_water_landcover.csv'
    # )
    # print("Scenario 5 complete.")


    # --- Scenario 6: Core + Climate + Topographical + Water + Land Cover + MODIS 16A2 Features ---
    # print("\n--- Running Scenario 6: Core + Climate + Topographical + Water + Land Cover + MODIS 16A2 Features ---")
    all_available_features_master_list.extend(modis16a2_features)

    # selected_features_s6 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s6 = train_df_full[selected_features_s6]
    # y_s6 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s6 = test_df_full[selected_features_s6]

    # X_train_s6, X_val_s6, y_train_s6, y_val_s6 = train_test_split(X_s6, y_s6, test_size=0.2, random_state=42)

    # _, _, final_predictions_s6, _ = train_and_save_models(
    #     X_train_s6, y_train_s6, X_val_s6, y_val_s6, X_test_final_s6,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario6_plus_modis16a2'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s6, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario6_plus_modis16a2.csv'
    # )
    # print("Scenario 6 complete.")

    # --- Scenario 7: Core + ... + MCD43A4 Features ---
    # print("\n--- Running Scenario 7: Core + ... + MCD43A4 Features ---")
    all_available_features_master_list.extend(mcd43a4_features)

    # selected_features_s7 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s7 = train_df_full[selected_features_s7]
    # y_s7 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s7 = test_df_full[selected_features_s7]

    # X_train_s7, X_val_s7, y_train_s7, y_val_s7 = train_test_split(X_s7, y_s7, test_size=0.2, random_state=42)

    # _, _, final_predictions_s7, _ = train_and_save_models(
    #     X_train_s7, y_train_s7, X_val_s7, y_val_s7, X_test_final_s7,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario7_plus_mcd43a4'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s7, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario7_plus_mcd43a4.csv'
    # )
    # print("Scenario 7 complete.")

    # --- Scenario 8: Core + ... + MOD09GA Features ---
    # print("\n--- Running Scenario 8: Core + ... + MOD09GA Features ---")
    all_available_features_master_list.extend(mod09ga_features)

    # selected_features_s8 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s8 = train_df_full[selected_features_s8]
    # y_s8 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s8 = test_df_full[selected_features_s8]

    # X_train_s8, X_val_s8, y_train_s8, y_val_s8 = train_test_split(X_s8, y_s8, test_size=0.2, random_state=42)

    # _, _, final_predictions_s8, _ = train_and_save_models(
    #     X_train_s8, y_train_s8, X_val_s8, y_val_s8, X_test_final_s8,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario8_plus_mod09ga'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s8, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario8_plus_mod09ga.csv'
    # )
    # print("Scenario 8 complete.")

    # --- Scenario 9: Core + ... + MOD13Q1 Features ---
    # print("\n--- Running Scenario 9: Core + ... + MOD13Q1 Features ---")
    all_available_features_master_list.extend(mod13q1_features)

    # selected_features_s9 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s9 = train_df_full[selected_features_s9]
    # y_s9 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s9 = test_df_full[selected_features_s9]

    # X_train_s9, X_val_s9, y_train_s9, y_val_s9 = train_test_split(X_s9, y_s9, test_size=0.2, random_state=42)

    # _, _, final_predictions_s9, _ = train_and_save_models(
    #     X_train_s9, y_train_s9, X_val_s9, y_val_s9, X_test_final_s9,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario9_plus_mod13q1'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s9, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario9_plus_mod13q1.csv'
    # )
    # print("Scenario 9 complete.")

    # --- Scenario 10: Core + ... + Landsat 8 Features ---
    # print("\n--- Running Scenario 10: Core + ... + Landsat 8 Features ---")
    all_available_features_master_list.extend(landsat8_features)

    # selected_features_s10 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s10 = train_df_full[selected_features_s10]
    # y_s10 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s10 = test_df_full[selected_features_s10]

    # X_train_s10, X_val_s10, y_train_s10, y_val_s10 = train_test_split(X_s10, y_s10, test_size=0.2, random_state=42)

    # _, _, final_predictions_s10, _ = train_and_save_models(
    #     X_train_s10, y_train_s10, X_val_s10, y_val_s10, X_test_final_s10,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario10_plus_landsat8'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s10, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario10_plus_landsat8.csv'
    # )
    # print("Scenario 10 complete.")

    # --- Scenario 11: Core + ... + Sentinel-1 Features ---
    # print("\n--- Running Scenario 11: Core + ... + Sentinel-1 Features ---")
    all_available_features_master_list.extend(sentinel1_features)

    # selected_features_s11 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    # X_s11 = train_df_full[selected_features_s11]
    # y_s11 = train_df_full[TARGET_COLUMNS]
    # X_test_final_s11 = test_df_full[selected_features_s11]

    # X_train_s11, X_val_s11, y_train_s11, y_val_s11 = train_test_split(X_s11, y_s11, test_size=0.2, random_state=42)

    # _, _, final_predictions_s11, _ = train_and_save_models(
    #     X_train_s11, y_train_s11, X_val_s11, y_val_s11, X_test_final_s11,
    #     TARGET_COLUMNS, models_dir='ensemble_models_scenario11_plus_sentinel1'
    # )
    # create_submission_file(
    #     test_df_full, final_predictions_s11, test_gap_df, TARGET_COLUMNS,
    #     output_filename='submission_scenario11_plus_sentinel1.csv'
    # )
    # print("Scenario 11 complete.")

    # --- Scenario 12: Core + ... + Sentinel-2 Features (All Features) ---
    print("\n--- Running Scenario 12: Core + ... + Sentinel-2 Features (All Features) ---")
    all_available_features_master_list.extend(sentinel2_features)
    all_available_features_master_list.extend(advanced_features_generated)
    

    selected_features_s12 = [col for col in all_available_features_master_list if col in train_df_full.columns]
    X_s12 = train_df_full[selected_features_s12]
    y_s12 = train_df_full[TARGET_COLUMNS]
    X_test_final_s12 = test_df_full[selected_features_s12]

    X_train_s12, X_val_s12, y_train_s12, y_val_s12 = train_test_split(X_s12, y_s12, test_size=0.2, random_state=42)

    _, _, final_predictions_s12, _ = train_and_save_models(
        X_train_s12, y_train_s12, X_val_s12, y_val_s12, X_test_final_s12,
        TARGET_COLUMNS, models_dir='ensemble_models_scenario12  _all_features'
    )
    create_submission_file(
        test_df_full, final_predictions_s12, test_gap_df, TARGET_COLUMNS,
        output_filename='submission_scenario12_all_features.csv'
    )
    print("Scenario 12 complete.")


    print("\nOverall script execution complete with multi-scenario ensemble model training.")
    print("Check respective directories for saved models and submission files for each scenario.")


--- Loading and Preprocessing Data ---
Handling initial missing values (mean imputation)...
Initial missing values handled.
Loading additional data from ../processed_data/...
MCD43A4 data loaded and merged successfully from ../processed_data/processed_modis_mcd43a4.parquet.
MOD09GA data loaded and merged successfully from ../processed_data/processed_modis_mod09ga.parquet.
MOD13Q1 data loaded and merged successfully from ../processed_data/processed_modis_mod13q1.parquet.
MOD16A2 data loaded and merged successfully from ../processed_data/processed_modis_mod16a2.parquet.
L8 data loaded and merged successfully from ../processed_data/processed_landsat_l8.parquet.
S1 data loaded and merged successfully from ../processed_data/processed_sentinel_s1.parquet.
S2 data loaded and merged successfully from ../processed_data/processed_sentinel_s2.parquet.

Performing initial NaN/Inf check and robust imputation...
Initial NaN/Inf check and imputation complete.

Applying advanced feature engineering to

In [ ]:
train_df_full.columns[80:90]

Index(['_MOD13Q1_MSI_m13q1', '_MOD16A2_ET', '_MOD16A2_PET',
       '_MOD16A2_ESI_mod16a2', '_MOD16A2_ETD_mod16a2', '8_LST_Celsius',
       '8_NDVI_ls8', '8_EVI_ls8', '8_SAVI_ls8', '8_NDWI_ls8'],
      dtype='object')